import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from pipeline.run_logic.ast_runner import load_step_module
phase_dir = Path(project_root) / 'pipeline' / 'phases' / 'phase_01_build_world--Lucas_Starkey'
load_step_module(phase_dir, phase_dir / 'steps' / 'step_04_build_graph.py')
load_step_module(phase_dir, phase_dir / 'steps' / 'step_07_build_field.py')
load_step_module(phase_dir, phase_dir / 'steps' / 'step_08_package_world.py')

from pipelineio.state import load_draft

run_id = os.environ.get('PIPELINE_RUN_ID', 'EXAMPLE_RUN_ID')
world = load_draft(f'../data/artifacts/runs/{run_id}/world/final_world.pkl')

print(f'World loaded with {len(world.hours())} hours.')


In [ ]:
# Find peak hour
hour_volumes = {}
for hk in world.hours():
    hour_volumes[hk] = sum(w.traveler_count for w in world.wap_timeslots.get(hk, {}).values())

peak_hour = max(hour_volumes, key=hour_volumes.get)
print(f"Peak hour: {peak_hour} with {hour_volumes[peak_hour]} transitions")

samples = world.get_flow(peak_hour)
print(f"{len(samples)} valid flow samples in the peak hour.")

xs = np.array([s.x for s in samples])
ys = np.array([s.y for s in samples])
us = np.array([s.dir_u for s in samples])
vs_arr = np.array([s.dir_v for s in samples])
mags = np.array([s.magnitude for s in samples])

# Normalize mag for colors
norm = mcolors.Normalize(vmin=mags.min(), vmax=mags.max())
cmap = cm.plasma
colors = cmap(norm(mags))

fig, ax = plt.subplots(figsize=(16, 12))
ax.set_facecolor('#0e1117')
fig.patch.set_facecolor('#0e1117')

# Draw the underlying graph edges (need wap coordinates)
# (We no longer have wap coordinates natively in graph unless we parse SVG. But we can skip it or just draw the points)
scale = 40
for i in range(len(xs)):
    ax.annotate(
        '', xy=(xs[i] + us[i] * scale, ys[i] + vs_arr[i] * scale),
        xytext=(xs[i], ys[i]),
        arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5 + norm(mags[i]) * 2.5),
        zorder=3
    )

scatter = ax.scatter(
    xs, ys, c=mags, cmap='plasma', s=20 + mags * 8,
    norm=norm, edgecolors='white', linewidths=0.3, zorder=4
)
plt.colorbar(scatter, ax=ax, label='Magnitude', fraction=0.03, pad=0.01)
ax.set_title(f'Campus Traffic Flow Field — Peak Hour {peak_hour}', color='white', fontsize=15, pad=12)
ax.invert_yaxis()
plt.show()
